In [1]:
# IMPORTO LAS LIBRERÍAS NECESARIAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')


**1. CARGA DEL CONJUNTO DE DATOS**

In [2]:
# CARGO LOS DATOS
url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"
df = pd.read_csv(url)

In [3]:
print("Shape:", df.shape)

Shape: (891, 3)


**2. ESTUDIO DE VARIABLES Y SU CONTENIDO**

En este conjunto de datos tendrá las siguientes variables:
- 'package_name': Nombre de la aplicación móvil (categórico)
- 'review': Comentario sobre la aplicación móvil (categórico)
- 'polarity': Variable de clase (0 o 1), siendo 0 un comentario negativo y 1 un comentario positivo (categórico-numérico)

In [4]:
df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   package_name  891 non-null    str  
 1   review        891 non-null    str  
 2   polarity      891 non-null    int64
dtypes: int64(1), str(2)
memory usage: 21.0 KB


In [6]:
df.describe(include='all')

,package_name,review,polarity
count,891,891,891.000000
unique,23,891,NaN
top,com.facebook.katana,privacy at least put some option appear offli...,NaN
freq,40,1,NaN
mean,NaN,NaN,0.344557
std,NaN,NaN,0.475490
min,NaN,NaN,0.000000
25%,NaN,NaN,0.000000
50%,NaN,NaN,0.000000
75%,NaN,NaN,1.000000


In [7]:
# Eliminar 'package_name'
df = df.drop(columns=["package_name"])

In [8]:
# Revisar valores nulos
print("Nulos por columna:")
print(df.isnull().sum())

Nulos por columna:
review      0
polarity    0
dtype: int64


In [9]:
# Eliminar filas con reviews nulas
df = df.dropna(subset=["review"])
print(f"\nRegistros tras limpiar nulos: {len(df)}")


Registros tras limpiar nulos: 891


In [10]:
# Limpiar texto: minúsculas y eliminar espacios
df["review"] = df["review"].str.strip().str.lower()

In [11]:
# Balance de clases
print("Distribución de clases (polarity):")
print(df["polarity"].value_counts())
print(f"\nPorcentaje de positivos: {df['polarity'].mean()*100:.1f}%")

Distribución de clases (polarity):
polarity
0    584
1    307
Name: count, dtype: int64

Porcentaje de positivos: 34.5%


In [12]:
# Train/test split
X = df["review"]
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

print(f"Train: {len(X_train)} muestras")
print(f"Test: {len(X_test)} muestras")

Train: 596 muestras
Test: 295 muestras


In [13]:
# Transformar el texto en una matriz de recuento de palabras
vec_model = CountVectorizer(stop_words="english")
X_train_vec = vec_model.fit_transform(X_train).toarray()
X_test_vec = vec_model.transform(X_test).toarray()

print(f"Vocabulario: {len(vec_model.vocabulary_)} palabras únicas")
print(f"Shape de X_train transformado: {X_train_vec.shape}")

Vocabulario: 2931 palabras únicas
Shape de X_train transformado: (596, 2931)


**3. COMPARAR LAS TRES IMPLEMENTACIONES DE NAIVE BAYES**

In [14]:
modelos = {
    "GaussianNB":    GaussianNB(),
    "MultinomialNB": MultinomialNB(),
    "BernoulliNB":   BernoulliNB()
}
resultados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train_vec, y_train)
    y_pred = modelo.predict(X_test_vec)
    accuracy = accuracy_score(y_test, y_pred)
    resultados[nombre] = accuracy
    print(f"{nombre}: accuracy = {accuracy}")

mejor = max(resultados, key=resultados.get)
print(f"\El mejor modelo es: {mejor} ({resultados[mejor]})")

GaussianNB: accuracy = 0.7728813559322034
MultinomialNB: accuracy = 0.8067796610169492
BernoulliNB: accuracy = 0.752542372881356
\El mejor modelo es: MultinomialNB (0.8067796610169492)


In [15]:
# Reporte del mejor modelo (MultinomialNB)
multinb = MultinomialNB()
multinb.fit(X_train_vec, y_train)
y_pred_mnb = multinb.predict(X_test_vec)

print("Classification Report - MultinomialNB")
print(classification_report(y_test, y_pred_mnb, target_names=["Negativo", "Positivo"]))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_mnb))

Classification Report - MultinomialNB
              precision    recall  f1-score   support

    Negativo       0.81      0.93      0.86       193
    Positivo       0.81      0.58      0.67       102

    accuracy                           0.81       295
   macro avg       0.81      0.75      0.77       295
weighted avg       0.81      0.81      0.80       295

Matriz de confusión:
[[179  14]
 [ 43  59]]


**4. OPTIMIZAR EL MODELO ANTERIOR**

**5. GUARDAR EL MODELO**

**6. OTRAS ALTERNATIVAS**